# Milestone 2
Saebastion Cole
10-18-26

## Data Transformation and Cleansing: ISO-3166 Countries

In [167]:
# Imports
import pandas as pd
from bs4 import BeautifulSoup

In [168]:
# Load the data
iso_df = pd.read_csv("data/ISO-3166Countries.csv", dtype=str, keep_default_na=False, na_values=[''])

## Transformation 1: normalize the column names
I will convert the naming to be one uniform format. For this I am picking kebab-casing.

In [169]:
# First data manipulation: fix column headers to all kebab casing
# I saw one of the column headers used underscores instead of hyphens.
iso_df.columns = iso_df.columns.str.replace('_', '-')
iso_df.head()

,name,alpha-2,alpha-3,country-code,iso-3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,Southern Asia,NaN,142,034,NaN
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150,154,NaN
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,Southern Europe,NaN,150,039,NaN
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,Northern Africa,NaN,002,015,NaN
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,Polynesia,NaN,009,061,NaN


## Transformation 2: Clean the region code to fill with 000 if empty

In [170]:
# Now I will fix the intermediate region code to default the values removing the NaN
print(f"Missing before: {iso_df['intermediate-region-code'].isna().sum()}")

iso_df['intermediate-region-code'] = iso_df['intermediate-region-code'].fillna('000')

iso_df.head()

Missing before: 144


,name,alpha-2,alpha-3,country-code,iso-3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,Southern Asia,NaN,142,034,000
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,NaN,150,154,000
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,Southern Europe,NaN,150,039,000
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,Northern Africa,NaN,002,015,000
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,Polynesia,NaN,009,061,000


## Transformation 3: Fix the region code column to say "None" if empty

In [171]:
# Now similarly I will fix the region code to default the values removing the NaN

iso_df['intermediate-region'] = iso_df['intermediate-region'].fillna('None')
  
iso_df.head()

,name,alpha-2,alpha-3,country-code,iso-3166-2,region,sub-region,intermediate-region,region-code,sub-region-code,intermediate-region-code
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,Southern Asia,None,142,034,000
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,Northern Europe,None,150,154,000
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,Southern Europe,None,150,039,000
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,Northern Africa,None,002,015,000
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,Polynesia,None,009,061,000


## Transformation 4: Reorder the dataframe columns

In [172]:
# Reorder the columns to make it easier to read
iso_df = iso_df[[
    'name', 'alpha-2', 'alpha-3', 'country-code',
    'iso-3166-2', 'region', 'region-code',
    'sub-region', 'sub-region-code',
    'intermediate-region', 'intermediate-region-code',
]]

iso_df.head()

,name,alpha-2,alpha-3,country-code,iso-3166-2,region,region-code,sub-region,sub-region-code,intermediate-region,intermediate-region-code
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,142,Southern Asia,034,None,000
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,150,Northern Europe,154,None,000
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,150,Southern Europe,039,None,000
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,002,Northern Africa,015,None,000
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,009,Polynesia,061,None,000


## Transformation 5: Add a column with the cleaned country names

In [173]:
# Make a column with a cleansed version of the country name. 
iso_df['name-cleansed'] = iso_df['name'].str.split(",").str[0]  # Remove any text after a comma


iso_df.loc[iso_df['alpha-2'] == 'KR', 'name-cleansed'] = 'South Korea'
iso_df.loc[iso_df['alpha-2'] == 'KP', 'name-cleansed'] = 'North Korea'

iso_df.head()


,name,alpha-2,alpha-3,country-code,iso-3166-2,region,region-code,sub-region,sub-region-code,intermediate-region,intermediate-region-code,name-cleansed
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,142,Southern Asia,034,None,000,Afghanistan
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,150,Northern Europe,154,None,000,Åland Islands
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,150,Southern Europe,039,None,000,Albania
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,002,Northern Africa,015,None,000,Algeria
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,009,Polynesia,061,None,000,American Samoa


## Transformation 6: Normalize the characters in the cleaned name removing accents

In [174]:
# Now I am going to cleanup the name columns to remove any romoving accents and unique characters.

iso_df['name-cleansed'] = iso_df['name-cleansed'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')
iso_df

,name,alpha-2,alpha-3,country-code,iso-3166-2,region,region-code,sub-region,sub-region-code,intermediate-region,intermediate-region-code,name-cleansed
0,Afghanistan,AF,AFG,004,ISO 3166-2:AF,Asia,142,Southern Asia,034,None,000,Afghanistan
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX,Europe,150,Northern Europe,154,None,000,Aland Islands
2,Albania,AL,ALB,008,ISO 3166-2:AL,Europe,150,Southern Europe,039,None,000,Albania
3,Algeria,DZ,DZA,012,ISO 3166-2:DZ,Africa,002,Northern Africa,015,None,000,Algeria
4,American Samoa,AS,ASM,016,ISO 3166-2:AS,Oceania,009,Polynesia,061,None,000,American Samoa
...,...,...,...,...,...,...,...,...,...,...,...,...
244,Wallis and Futuna,WF,WLF,876,ISO 3166-2:WF,Oceania,009,Polynesia,061,None,000,Wallis and Futuna
245,Western Sahara,EH,ESH,732,ISO 3166-2:EH,Africa,002,Northern Africa,015,None,000,Western Sahara
246,Yemen,YE,YEM,887,ISO 3166-2:YE,Asia,142,Western Asia,145,None,000,Yemen
247,Zambia,ZM,ZMB,894,ISO 3166-2:ZM,Africa,002,Sub-Saharan Africa,202,Eastern Africa,014,Zambia


## Ethical Concerns About Data Wrangling this Data
For my final project I selected a very structured and standard data set for my file based data. Since this is closely following the ISO-3166 Country Code standards drastic edits to the data will only negatively effect its accuracy. That is why I only added new data columns and organized the formatting of it all instead of manipulating what is there. Following the ISO standard is the best way to follow the internationally agreed on naming and standards. One of the thoughts I had while manipulating the data was to make it as easy as I can to work on later. I made sure the dataset was complete with the concept of binning in mind. The data was ethically acquired from a publicly availible repo on GitHub and also I skimmed it to verify its integrity compared to the ISO.org site itself. Ethical concerns were mitigated by using this standard data to begin with. These are the agreed upon conventions to following them is best practice.